<a href="https://colab.research.google.com/github/ShabahangShayegan/TP2_codage_video_neuronal/blob/main/TP2_codage_video_neuronal_etudiant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP2 — Codage vidéo neuronal : du mouvement par blocs au codage conditionnel

**Cours : Machine Learning appliqué à la compression vidéo — EPITA, majeure IA — Séance 2 (3h)**

Au semestre précédent, vous avez vu la compensation de mouvement **par blocs, par nœuds et par maillage**. Dans ce TP, vous allez :

| Partie | Durée | Contenu |
|---|---|---|
| **A** | ~45 min | Estimation de mouvement : **block matching** contre flot optique appris (**RAFT**) |
| **B** | ~40 min | Un codec vidéo neuronal à **codage de résidu** : **ScaleSpaceFlow** (Google, CVPR 2020) |
| **C** | ~75 min | Protocole commun **low-delay** : **x265**, **VVenC (VVC)**, ScaleSpaceFlow et **DCVC-RT** (Microsoft, CVPR 2025, **codage conditionnel**), courbes R-D, BD-rate, **dérive de qualité dans le GOP** |

Tous les résultats sont **sauvegardés au fur et à mesure** (sur votre Google Drive si possible) : si la session Colab est coupée, il suffit de tout ré-exécuter, les calculs déjà faits sont relus et non refaits.

---
**Avant de commencer**
- `Fichier > Enregistrer une copie dans Drive`.
- `Exécution > Modifier le type d'exécution > GPU (T4)`. **DCVC-RT exige un GPU.**
- Les cellules **✏️ Exercice** sont à compléter ; les **❓ Questions** vont dans le rapport.

## 0. Configuration, installation, ressources

**Mode `FAST` (par défaut)** : séquences UVG sous-échantillonnées en 960×540, 32 images (une période intra). Tient dans le temps du TP sur un GPU T4.
**Mode `FULL`** : 1920×1080, 96 images (trois périodes intra), conditions proches des publications. Plus long.

Les séquences, les poids de DCVC-RT et l'exécutable VVenC sont **pré-téléchargés** sur un dépôt Hugging Face préparé par l'enseignant (`ASSETS_REPO`). Des solutions de repli existent pour chacun.

In [ ]:
# ===================== Paramètres du TP =====================
MODE          = "FAST"           # "FAST" (960x540, 32 images) ou "FULL" (1920x1080, 96 images)
ASSETS_REPO   = "ShabahangShayegan/epita-ml-video-assets"   # dépôt Hugging Face (type dataset) de l'enseignant
SEQUENCES     = ["Jockey", "Bosphorus"]                     # disponibles : Jockey, Bosphorus, HoneyBee, ReadySetGo
INTRA_PERIOD  = 32               # période intra commune à tous les codecs
USE_DRIVE     = True             # sauvegarde des résultats sur Google Drive (reprise après coupure)
USE_PRECOMPUTED_IF_MISSING = True  # utiliser les résultats de l'enseignant pour un codec qui n'a pas pu tourner
RUN = {"x265": True, "VVenC": True, "SSF": True, "DCVC-RT": True}   # désactivez un codec si le temps manque
# ============================================================

W, H     = (960, 540) if MODE == "FAST" else (1920, 1080)
N_FRAMES = 32 if MODE == "FAST" else 96
FPS      = 120

In [ ]:
!pip install -q --no-deps compressai==1.2.8 torch-geometric pytorch-msssim einops

In [ ]:
import os, sys, io, re, json, glob, time, shutil, subprocess, tempfile, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from scipy.interpolate import PchipInterpolator

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device :", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "PAS DE GPU : DCVC-RT sera remplacé par les résultats pré-calculés")
torch.manual_seed(0); np.random.seed(0)
plt.rcParams["figure.dpi"] = 110
PRETRAINED = True          # poids pré-entraînés (CompressAI, torchvision)

IN_COLAB = "google.colab" in sys.modules
WORK = "tp2_work"
if IN_COLAB:
    WORK = "/content/tp2_work"
    if USE_DRIVE:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            WORK = "/content/drive/MyDrive/EPITA_TP2_video"
        except Exception as e:
            print("Drive non monté, résultats en local seulement :", e)
for sub in ["data", "results", "bin", "assets", "tmp"]:
    os.makedirs(os.path.join(WORK, sub), exist_ok=True)
# Fichiers volumineux et liens symboliques : disque local de la session (Google Drive ne gère pas les liens)
LOCAL = "/content/tp2_local" if IN_COLAB else os.path.join(WORK, "local")
os.makedirs(LOCAL, exist_ok=True)
print("Dossier de travail :", WORK, "| mode", MODE, f"({W}x{H}, {N_FRAMES} images)")

def sh(cmd, check=True, **kw):
    # Lance une commande shell et renvoie sa sortie (erreur lisible si échec)
    r = subprocess.run(cmd, shell=isinstance(cmd, str), capture_output=True, text=True, **kw)
    if check and r.returncode != 0:
        raise RuntimeError(f"Échec ({r.returncode}) : {cmd}\n{r.stdout[-2000:]}\n{r.stderr[-2000:]}")
    return r.stdout + r.stderr

def hf_get(path_in_repo):
    # Télécharge un fichier du dépôt de ressources de l'enseignant (None si indisponible)
    try:
        from huggingface_hub import hf_hub_download
        return hf_hub_download(ASSETS_REPO, path_in_repo, repo_type="dataset",
                               local_dir=os.path.join(WORK, "assets"))
    except Exception as e:
        print(f"  [ressource indisponible] {path_in_repo} : {type(e).__name__}")
        return None

## 1. Données : séquences UVG en YUV 4:2:0

Les séquences **UVG** (Ultra Video Group, Tampere ; licence CC BY-NC 3.0) sont, avec les séquences HEVC et MCL-JCV, la référence des articles de codage vidéo neuronal. Elles sont en **YUV 4:2:0 8 bits** : une image = un plan de luminance Y (W×H) et deux plans de chrominance U, V (W/2 × H/2).

Ordre de recherche : cache local/Drive → dépôt de l'enseignant → site UVG (archive 7z, extraction des premières images) → séquence synthétique de secours.

In [ ]:
UVG_URL = "https://ultravideo.fi/video/{name}_1920x1080_120fps_420_8bit_YUV_RAW.7z"
frame_bytes = lambda w, h: w * h * 3 // 2

def read_yuv420(path, w, h, n, start=0):
    fb = frame_bytes(w, h)
    frames = []
    with open(path, "rb") as f:
        f.seek(start * fb)
        for _ in range(n):
            buf = f.read(fb)
            if len(buf) < fb: break
            a = np.frombuffer(buf, np.uint8)
            Y = a[:w * h].reshape(h, w)
            U = a[w * h:w * h * 5 // 4].reshape(h // 2, w // 2)
            V = a[w * h * 5 // 4:].reshape(h // 2, w // 2)
            frames.append((Y, U, V))
    return frames

def write_yuv420(path, frames):
    with open(path, "wb") as f:
        for Y, U, V in frames:
            f.write(Y.tobytes()); f.write(U.tobytes()); f.write(V.tobytes())

def downscale_yuv(src, dst, w0, h0, w1, h1, n):
    sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{w0}x{h0}",
        "-i", src, "-frames:v", str(n), "-vf", f"scale={w1}:{h1}:flags=lanczos",
        "-f", "rawvideo", "-pix_fmt", "yuv420p", dst])

def fetch_from_uvg_site(name, dst_1080, n):
    # Télécharge l'archive 7z officielle et n'extrait que les n premières images (flux sur stdout)
    arc = os.path.join(LOCAL, f"{name}.7z")
    if not os.path.exists(arc):
        print(f"  téléchargement de l'archive UVG {name} (~0,7 Go)…")
        sh(["wget", "-q", "-O", arc, UVG_URL.format(name=name)])
    if shutil.which("7z") is None:
        sh("apt-get -qq install -y p7zip-full")
    need = n * frame_bytes(1920, 1080)
    p = subprocess.Popen(["7z", "e", "-so", arc], stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    data = p.stdout.read(need); p.kill()
    assert len(data) == need, "extraction incomplète"
    with open(dst_1080, "wb") as f: f.write(data)

def synthetic_sequence(name, w, h, n):
    # Repli : panoramique sur une image + objet en mouvement (permet de faire tourner le TP sans réseau)
    from skimage import data as skd
    base = [skd.astronaut, skd.chelsea, skd.rocket, skd.coffee][sum(map(ord, name)) % 4]()
    big = torch.from_numpy(base).permute(2, 0, 1)[None].float() / 255
    big = F.interpolate(big, size=(h + 4 * n, w + 6 * n), mode="bicubic", align_corners=False).clamp(0, 1)
    obj = torch.from_numpy(skd.cat()).permute(2, 0, 1)[None].float() / 255
    oh, ow = h // 3, w // 4
    obj = F.interpolate(obj, size=(oh, ow), mode="bilinear", align_corners=False)
    frames = []
    for t in range(n):
        fr = big[..., 2 * t:2 * t + h, 3 * t:3 * t + w].clone()
        y0, x0 = h // 3 + int(20 * np.sin(t / 5)), int(w * 0.1 + 9 * t) % (w - ow)
        fr[..., y0:y0 + oh, x0:x0 + ow] = obj
        frames.append(rgb_to_yuv420(fr))
    return frames

def get_sequence(name):
    dst = os.path.join(WORK, "data", f"{name}_{W}x{H}_{N_FRAMES}f.yuv")
    if os.path.exists(dst) and os.path.getsize(dst) >= N_FRAMES * frame_bytes(W, H):
        return dst
    print(f"Préparation de {name}…")
    src = hf_get(f"uvg/{name}_{W}x{H}_96f.yuv")
    if src is not None:
        write_yuv420(dst, read_yuv420(src, W, H, N_FRAMES)); return dst
    try:
        tmp1080 = os.path.join(LOCAL, f"{name}_1920x1080_{N_FRAMES}f.yuv")
        if not os.path.exists(tmp1080):
            fetch_from_uvg_site(name, tmp1080, N_FRAMES)
        if (W, H) == (1920, 1080):
            write_yuv420(dst, read_yuv420(tmp1080, W, H, N_FRAMES))
        else:
            downscale_yuv(tmp1080, dst, 1920, 1080, W, H, N_FRAMES)
        return dst
    except Exception as e:
        print("  site UVG indisponible :", e)
    print(f"  !!! {name} : séquence SYNTHÉTIQUE de secours (les conclusions ne valent pas pour UVG)")
    write_yuv420(dst, synthetic_sequence(name, W, H, N_FRAMES))
    return dst

### Conversions de couleur et métrique du protocole

Les codecs classiques et DCVC-RT travaillent **directement en YUV 4:2:0**. ScaleSpaceFlow et RAFT attendent du **RGB** : on convertit avec la matrice **BT.709 pleine échelle** (recommandation des conditions de test JPEG AI et DCVC).

La métrique du protocole est le **PSNR YUV pondéré** calculé image par image puis moyenné sur la séquence :
$$\mathrm{PSNR}_{YUV} = \frac{6\,\mathrm{PSNR}_Y + \mathrm{PSNR}_U + \mathrm{PSNR}_V}{8}$$
Le débit est en **bits par pixel** : $\text{bpp} = \dfrac{\text{bits totaux}}{N \cdot W \cdot H}$.

In [ ]:
KR, KB = 0.2126, 0.0722   # BT.709

def yuv420_to_rgb(Y, U, V):
    # uint8 YUV 4:2:0 -> tenseur 1x3xHxW dans [0,1] (chroma suréchantillonnée au plus proche voisin)
    y = torch.from_numpy(Y.astype(np.float32) / 255)
    u = torch.from_numpy(U.astype(np.float32) / 255 - 0.5).repeat_interleave(2, 0).repeat_interleave(2, 1)
    v = torch.from_numpy(V.astype(np.float32) / 255 - 0.5).repeat_interleave(2, 0).repeat_interleave(2, 1)
    r = y + 2 * (1 - KR) * v
    b = y + 2 * (1 - KB) * u
    g = (y - KR * r - KB * b) / (1 - KR - KB)
    return torch.stack([r, g, b])[None].clamp(0, 1)

def rgb_to_yuv420(x):
    # tenseur 1x3xHxW dans [0,1] -> (Y, U, V) uint8 4:2:0 (chroma moyennée sur 2x2)
    x = x.detach().float().cpu().clamp(0, 1)
    r, g, b = x[0, 0], x[0, 1], x[0, 2]
    y = KR * r + (1 - KR - KB) * g + KB * b
    u = (b - y) / (2 * (1 - KB)) + 0.5
    v = (r - y) / (2 * (1 - KR)) + 0.5
    u = F.avg_pool2d(u[None, None], 2)[0, 0]; v = F.avg_pool2d(v[None, None], 2)[0, 0]
    q = lambda t: (t.clamp(0, 1) * 255).round().byte().numpy()
    return q(y), q(u), q(v)

def show_yuv(frame, ax=None, title=None):
    ax = ax or plt.gca()
    ax.imshow(yuv420_to_rgb(*frame)[0].permute(1, 2, 0).numpy()); ax.axis("off")
    if title: ax.set_title(title, fontsize=9)

### ✏️ Exercice 1 — PSNR YUV du protocole

Complétez `psnr_yuv420(src, rec)` : `src` et `rec` sont des triplets `(Y, U, V)` de tableaux `uint8`. Renvoyez `(psnr_y, psnr_u, psnr_v, psnr_yuv)` avec la pondération 6:1:1. Plafonnez chaque PSNR à 100 dB (cas d'une image identique).

In [ ]:
def psnr_plane(a, b):
    # TODO : PSNR d'un plan uint8 (crête 255), plafonné à 100 dB
    raise NotImplementedError("Exercice 1")

def psnr_yuv420(src, rec):
    # TODO : (psnr_y, psnr_u, psnr_v, psnr_yuv) avec psnr_yuv = (6*Y + U + V) / 8
    raise NotImplementedError("Exercice 1")

In [ ]:
SEQ_PATHS = {s: get_sequence(s) for s in SEQUENCES}
SEQ = {s: read_yuv420(p, W, H, N_FRAMES) for s, p in SEQ_PATHS.items()}

fig, axes = plt.subplots(len(SEQUENCES), 3, figsize=(13, 2.6 * len(SEQUENCES)), squeeze=False)
for row, s in zip(axes, SEQUENCES):
    for ax, t in zip(row, [0, N_FRAMES // 2, N_FRAMES - 1]):
        show_yuv(SEQ[s][t], ax, f"{s} — image {t}")
plt.tight_layout(); plt.show()

# Sanity check : le passage YUV -> RGB -> YUV (sans compression) n'est pas sans perte !
f0 = SEQ[SEQUENCES[0]][0]
print("Aller-retour YUV->RGB->YUV : PSNR_YUV = %.2f dB (plafond pour un codec qui travaille en RGB)"
      % psnr_yuv420(f0, rgb_to_yuv420(yuv420_to_rgb(*f0)))[3])

---
# Partie A — Estimation de mouvement : blocs contre flot optique appris

On prend deux images d'une séquence, une **référence** $I_{t-D}$ et l'image **courante** $I_t$, et on cherche à prédire $I_t$ à partir de $I_{t-D}$ :
- **block matching** (H.264/HEVC) : un vecteur par bloc, recherche exhaustive minimisant la SAD ;
- **RAFT** (Teed & Deng, ECCV 2020, pré-entraîné, fourni par torchvision) : un vecteur **par pixel**.

Dans les deux cas, la prédiction est un **warping arrière** : $\hat I_t(p) = I_{t-D}\big(p + f(p)\big)$, où $f$ est le champ de mouvement de l'image courante vers la référence.

In [ ]:
SEQ_A, D = SEQUENCES[0], 4
HA, WA = H // 16 * 16, W // 16 * 16                 # on recadre à un multiple de 16 pour les blocs
crop_frame = lambda fr: (fr[0][:HA, :WA], fr[1][:HA // 2, :WA // 2], fr[2][:HA // 2, :WA // 2])
ref_f, cur_f = crop_frame(SEQ[SEQ_A][0]), crop_frame(SEQ[SEQ_A][D])
to_y = lambda fr: torch.from_numpy(fr[0].astype(np.float32) / 255).to(device)
ref_y, cur_y = to_y(ref_f), to_y(cur_f)

fig, ax = plt.subplots(1, 3, figsize=(15, 3.4))
show_yuv(ref_f, ax[0], f"référence (image 0)"); show_yuv(cur_f, ax[1], f"courante (image {D})")
ax[2].imshow((cur_y - ref_y).abs().cpu(), cmap="magma", vmax=0.3); ax[2].set_title("|différence| (prédiction sans mouvement)"); ax[2].axis("off")
plt.tight_layout(); plt.show()

### ✏️ Exercice 2 — Warping arrière

Complétez `warp(ref, flow)` : `ref` est un tenseur `1×C×H×W`, `flow` un tenseur `1×2×H×W` en **pixels**, canal 0 = déplacement horizontal $f_x$, canal 1 = vertical $f_y$. Renvoyez $\hat I(p) = \text{ref}(p + f(p))$ avec `F.grid_sample` (interpolation bilinéaire, `padding_mode="border"`, `align_corners=True`).

*Rappel : `grid_sample` attend des coordonnées normalisées dans $[-1, 1]$ : $x_{norm} = 2x/(W-1) - 1$.*

In [ ]:
def warp(ref, flow):
    _, _, h, w = ref.shape
    # TODO : grille de coordonnées (meshgrid), ajout du flot, normalisation dans [-1,1], grid_sample
    raise NotImplementedError("Exercice 2")

### ✏️ Exercice 3 — Block matching par recherche exhaustive

Complétez `block_matching(cur, ref, B, R)` sur la luminance (`cur`, `ref` : tenseurs `H×W`). Pour chaque bloc $B\times B$ de l'image courante, cherchez le déplacement entier $(d_x, d_y) \in [-R, R]^2$ qui minimise la **SAD** avec la référence.

Astuce vectorisée (déjà en place) : au lieu de boucler sur les blocs, on boucle sur les $(2R+1)^2$ déplacements ; pour chacun, on calcule la SAD de **tous les blocs d'un coup** avec `F.avg_pool2d`. Renvoyez un tenseur `2 × H/B × W/B` (canal 0 = $d_x$, canal 1 = $d_y$) et la SAD minimale par bloc.

In [ ]:
def block_matching(cur, ref, B=16, R=16):
    h, w = cur.shape
    ref_p = F.pad(ref[None, None], (R, R, R, R), mode="replicate")[0, 0]
    best = torch.full((h // B, w // B), float("inf"), device=cur.device)
    mv = torch.zeros((2, h // B, w // B), device=cur.device)
    for dy in range(-R, R + 1):
        for dx in range(-R, R + 1):
            # TODO : 1) image de référence décalée de (dx, dy) (utilisez ref_p)
            #        2) SAD par bloc avec F.avg_pool2d
            #        3) mise à jour de best et mv là où la SAD est meilleure
            raise NotImplementedError("Exercice 3")
    return mv, best

In [ ]:
def mv_to_flow(mv, B):
    return mv.repeat_interleave(B, 1).repeat_interleave(B, 2)[None]   # champ dense constant par bloc

R_SEARCH = 24 if MODE == "FAST" else 48
results_A = {}
pred_none = ref_y
for B in [16, 8]:
    t0 = time.time()
    mv, _ = block_matching(cur_y, ref_y, B, R_SEARCH)
    flow = mv_to_flow(mv, B)
    results_A[f"blocs {B}x{B}"] = dict(flow=flow, mv=mv, pred=warp(ref_y[None, None], flow)[0, 0], t=time.time() - t0)
    print(f"block matching {B}x{B}, R={R_SEARCH} : {time.time()-t0:.2f} s")

### RAFT : flot optique dense appris

RAFT est entraîné à **estimer le mouvement réel** (sur des données synthétiques puis réelles), pas à minimiser une erreur de prédiction ni un débit. Nous l'utilisons tel quel, pré-entraîné.

In [ ]:
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.utils import flow_to_image

raft_w = Raft_Large_Weights.DEFAULT
raft = raft_large(weights=raft_w if PRETRAINED else None, progress=False).eval().to(device)
raft_tf = raft_w.transforms()

def raft_flow(cur_frame, ref_frame):
    # Flot de l'image courante vers la référence : cur(p) ≈ ref(p + f(p))
    a, b = yuv420_to_rgb(*cur_frame).to(device), yuv420_to_rgb(*ref_frame).to(device)
    a, b = raft_tf(a, b)
    with torch.no_grad():
        return raft(a, b, num_flow_updates=12)[-1]

t0 = time.time()
flow_raft = raft_flow(cur_f, ref_f)
results_A["RAFT (dense)"] = dict(flow=flow_raft, pred=warp(ref_y[None, None], flow_raft)[0, 0], t=time.time() - t0)
print(f"RAFT : {time.time()-t0:.2f} s, flot {tuple(flow_raft.shape)}")

### Qualité de prédiction et coût du mouvement

Une bonne prédiction ne suffit pas : **le décodeur doit recevoir le mouvement**. On estime son coût par l'**entropie empirique** des vecteurs (borne inférieure du codage entropique sans contexte) :
- blocs : un vecteur entier par bloc ;
- RAFT : un vecteur par pixel, quantifié au **quart de pixel** comme en HEVC.

In [ ]:
def entropy_bits(symbols):
    _, counts = np.unique(symbols, axis=0, return_counts=True)
    p = counts / counts.sum()
    return -(counts * np.log2(p)).sum()

def psnr_t(a, b):
    return (10 * torch.log10(1 / F.mse_loss(a, b))).item()

rows = [dict(méthode="sans mouvement", psnr_pred_Y=psnr_t(pred_none, cur_y), bits_mvt=0)]
for name, r in results_A.items():
    if "blocs" in name:
        vecs = r["mv"].reshape(2, -1).T.cpu().numpy()
    else:
        vecs = torch.round(r["flow"][0] * 4).reshape(2, -1).T.cpu().numpy()   # 1/4 de pixel
    rows.append(dict(méthode=name, psnr_pred_Y=psnr_t(r["pred"], cur_y), bits_mvt=entropy_bits(vecs)))
tabA = pd.DataFrame(rows).set_index("méthode")
tabA["bpp_mvt"] = tabA.bits_mvt / (HA * WA)
display(tabA.round(3))

fig, axes = plt.subplots(3, 4, figsize=(16, 8))
names = ["sans mouvement"] + list(results_A)
for j, name in enumerate(names):
    pred = pred_none if name == "sans mouvement" else results_A[name]["pred"]
    if name == "sans mouvement":
        axes[0, j].imshow(np.zeros((HA, WA, 3))); axes[0, j].set_title("flot nul", fontsize=9)
    else:
        axes[0, j].imshow(flow_to_image(results_A[name]["flow"][0].cpu()).permute(1, 2, 0)); axes[0, j].set_title(f"flot : {name}", fontsize=9)
    axes[1, j].imshow(pred.cpu(), cmap="gray"); axes[1, j].set_title(f"prédiction — {psnr_t(pred, cur_y):.2f} dB", fontsize=9)
    axes[2, j].imshow((cur_y - pred).abs().cpu(), cmap="magma", vmax=0.25); axes[2, j].set_title("|résidu|", fontsize=9)
for a in axes.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

### Influence de la distance temporelle

In [ ]:
dists = [d for d in [1, 2, 4, 8, 16] if d < N_FRAMES]
curve = {k: [] for k in ["sans mouvement", "blocs 16x16", "RAFT (dense)"]}
for d in dists:
    c = crop_frame(SEQ[SEQ_A][d]); cy = to_y(c)
    curve["sans mouvement"].append(psnr_t(ref_y, cy))
    mv, _ = block_matching(cy, ref_y, 16, R_SEARCH)
    curve["blocs 16x16"].append(psnr_t(warp(ref_y[None, None], mv_to_flow(mv, 16))[0, 0], cy))
    curve["RAFT (dense)"].append(psnr_t(warp(ref_y[None, None], raft_flow(c, ref_f))[0, 0], cy))
plt.figure(figsize=(6.5, 4))
for k, v in curve.items(): plt.plot(dists, v, "o-", label=k)
plt.xscale("log", base=2); plt.xticks(dists, dists); plt.xlabel("distance D (images)"); plt.ylabel("PSNR de prédiction Y (dB)")
plt.grid(alpha=.3); plt.legend(); plt.title(f"{SEQ_A} : qualité de prédiction vs distance"); plt.show()

**❓ Questions — Partie A**
1. Comparez les prédictions par blocs et par RAFT : où chacune échoue-t-elle ? Repérez les **occlusions/désocclusions** (zones découvertes par un objet en mouvement) : un warping, quel qu'il soit, peut-il les prédire ?
2. RAFT prédit-il toujours mieux que le block matching ? Pourquoi un flot qui estime le **mouvement réel** n'est-il pas forcément celui qui **minimise l'erreur de prédiction** ?
3. Comparez le coût du mouvement (`bpp_mvt`) au débit total typique d'une vidéo (de l'ordre de 0,02 à 0,1 bpp). Conclusion : peut-on transmettre un flot dense tel quel ? Quelles sont les deux solutions que vous allez voir (Partie B et Partie C) ?
4. Faites le lien avec le cours du semestre précédent : à quoi ressemble un champ par maillage (*mesh-based*) par rapport à ces deux extrêmes (un vecteur par bloc / un vecteur par pixel) ?

---
# Partie B — ScaleSpaceFlow : un codec vidéo neuronal à codage de résidu

**ScaleSpaceFlow** (Agustsson et al., CVPR 2020) reprend la structure du codage hybride avec trois autoencodeurs :
1. un autoencodeur **de mouvement** : à partir de $(x_t, \hat x_{t-1})$, il code un flot **et un champ d'échelle** $s$ ;
2. la **prédiction** $\tilde x_t$ : warping dans un *volume d'échelles* (la référence floutée à plusieurs niveaux). Là où le mouvement est incertain, le réseau choisit une version **floue** de la référence plutôt qu'un warping faux ;
3. un autoencodeur **de résidu** code $x_t - \tilde x_t$.

C'est exactement le schéma « prédiction + résidu » d'un codec classique, mais **chaque bloc est appris de bout en bout** pour minimiser $R + \lambda D$. Le modèle de CompressAI travaille en **RGB**.

In [ ]:
from compressai.zoo import ssf2020

Q_SSF = 5
ssf = ssf2020(quality=Q_SSF, pretrained=PRETRAINED).eval().to(device)
ssf.update(force=True)

PAD_SSF = 128  # sous-échantillonnage total du réseau (latents x hyper-latents) : dimensions multiples de 128
def pad_to(x, p):
    h, w = x.shape[-2:]
    return F.pad(x, (0, (-w) % p, 0, (-h) % p), mode="replicate")

def ssf_inter_debug(net, x_cur, x_ref):
    # Réplique de net.encode_inter qui renvoie aussi les grandeurs intermédiaires
    y_motion = net.motion_encoder(torch.cat((x_cur, x_ref), dim=1))
    y_motion_hat, out_motion = net.motion_hyperprior.compress(y_motion)
    motion_info = net.motion_decoder(y_motion_hat)
    x_pred = net.forward_prediction(x_ref, motion_info)
    y_res_hat, out_res = net.res_hyperprior.compress(net.res_encoder(x_cur - x_pred))
    x_res_hat = net.res_decoder(torch.cat((y_res_hat, y_motion_hat), dim=1))
    flow, scale = motion_info.chunk(2, dim=1)
    h, w = x_cur.shape[-2:]
    flow_px = torch.stack([flow[:, 0] * w / 2, flow[:, 1] * h / 2], 1)   # coordonnées normalisées -> pixels
    strings = {"motion": out_motion["strings"], "residual": out_res["strings"]}
    shapes = {"motion": out_motion["shape"], "residual": out_res["shape"]}
    return x_pred + x_res_hat, strings, shapes, dict(pred=x_pred, flow=flow_px, scale=scale, res=x_cur - x_pred, res_hat=x_res_hat)

### ✏️ Exercice 4 — Compter les bits d'une image codée

Les flux de CompressAI sont des structures imbriquées (dictionnaires de listes de listes de `bytes`). Écrivez `count_bits(obj)` qui renvoie le nombre total de **bits**, quel que soit le niveau d'imbrication.

In [ ]:
def count_bits(obj):
    # TODO : récursion sur bytes / dict / list / tuple
    raise NotImplementedError("Exercice 4")

In [ ]:
G = min(16, N_FRAMES)
frames_rgb = [pad_to(yuv420_to_rgb(*f), PAD_SSF).to(device) for f in SEQ[SEQ_A][:G]]
hh, ww = H, W
log_B, debug = [], {}
with torch.no_grad():
    x_ref, out_k = ssf.encode_keyframe(frames_rgb[0])
    rec0 = ssf.decode_keyframe(out_k["strings"], out_k["shape"])
    log_B.append(dict(t=0, type="I", bits_motion=0, bits_res=count_bits(out_k["strings"]),
                      psnr=psnr_yuv420(SEQ[SEQ_A][0], rgb_to_yuv420(rec0[..., :hh, :ww]))[3]))
    dec_ref = rec0
    for t in range(1, G):
        x_rec, strings, shapes, dbg = ssf_inter_debug(ssf, frames_rgb[t], x_ref)
        x_ref = x_rec
        dec_ref = ssf.decode_inter(dec_ref, strings, shapes)       # vrai décodage à partir du flux
        if t == D: debug = dbg
        log_B.append(dict(t=t, type="P", bits_motion=count_bits(strings["motion"]), bits_res=count_bits(strings["residual"]),
                          psnr=psnr_yuv420(SEQ[SEQ_A][t], rgb_to_yuv420(dec_ref[..., :hh, :ww]))[3]))
print("écart encodeur/décodeur sur la dernière image : %.2e (doit être ~0)" % (x_ref - dec_ref).abs().max().item())
logB = pd.DataFrame(log_B)
display(logB.head())

In [ ]:
if not debug:  # si D >= G, on prend la dernière image P
    debug = dbg
crop = lambda t: t[0, :, :hh, :ww].detach().cpu()
fig, ax = plt.subplots(2, 4, figsize=(17, 6.3))
ax[0, 0].imshow(crop(frames_rgb[D if D < G else G - 1]).permute(1, 2, 0)); ax[0, 0].set_title("image courante")
ax[0, 1].imshow(flow_to_image(crop(debug["flow"])).permute(1, 2, 0)); ax[0, 1].set_title("flot décodé (SSF)")
im = ax[0, 2].imshow(crop(debug["scale"])[0], cmap="viridis"); ax[0, 2].set_title("champ d'échelle s (flou)"); plt.colorbar(im, ax=ax[0, 2], fraction=.03)
ax[0, 3].imshow(flow_to_image(results_A["RAFT (dense)"]["flow"][0].cpu()).permute(1, 2, 0)); ax[0, 3].set_title(f"RAFT, même paire (D={D})" if D < G else "RAFT (Partie A)")
ax[1, 0].imshow(crop(debug["pred"]).clamp(0, 1).permute(1, 2, 0)); ax[1, 0].set_title("prédiction x̃")
ax[1, 1].imshow(crop(debug["res"]).abs().mean(0), cmap="magma", vmax=.2); ax[1, 1].set_title("|résidu| à coder")
ax[1, 2].imshow(crop(debug["res_hat"]).abs().mean(0), cmap="magma", vmax=.2); ax[1, 2].set_title("|résidu décodé|")
ax[1, 3].imshow(crop(debug["pred"] + debug["res_hat"]).clamp(0, 1).permute(1, 2, 0)); ax[1, 3].set_title("reconstruction")
for a in ax.ravel(): a.axis("off")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 3.6))
ax[0].bar(logB.t, logB.bits_res / 1e3, label="résidu (ou intra)"); ax[0].bar(logB.t, logB.bits_motion / 1e3, bottom=logB.bits_res / 1e3, label="mouvement")
ax[0].set_xlabel("image"); ax[0].set_ylabel("kbits"); ax[0].set_yscale("log"); ax[0].legend(); ax[0].set_title("Bits par image")
ax[1].plot(logB.t, logB.psnr, "o-"); ax[1].set_xlabel("image"); ax[1].set_ylabel("PSNR YUV (dB)"); ax[1].set_title("Qualité le long du GOP"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()
print(f"Part du mouvement dans les images P : {100 * logB.bits_motion[1:].sum() / (logB.bits_motion[1:] + logB.bits_res[1:]).sum():.1f} %")

**❓ Questions — Partie B**
5. Comparez le flot décodé de SSF au flot de RAFT. Pourquoi le flot de SSF est-il plus lisse ? (Pensez au terme de débit dans la fonction de coût.)
6. Où le champ d'échelle $s$ est-il élevé ? Quel problème de la Partie A cela résout-il, et quel est l'équivalent dans un codec classique ?
7. Quelle fraction du débit des images P est consacrée au mouvement ? Comparez à l'ordre de grandeur de la Partie A.
8. Le PSNR évolue-t-il le long du GOP ? Expliquez le mécanisme de **propagation d'erreur** dans un codage prédictif où la référence est une image *décodée*.

---
# Partie C — Protocole commun : x265, VVenC, ScaleSpaceFlow, DCVC-RT

Comparer des codecs n'a de sens que si **les conditions de test sont identiques**. Nous suivons les conditions recommandées par l'équipe DCVC (Microsoft) pour comparer codecs neuronaux et classiques :

| | Protocole commun |
|---|---|
| Source | YUV 4:2:0 8 bits, **sans recadrage** (les codecs complètent par padding si besoin) |
| Structure | **low-delay** : chaque image n'utilise que des images passées, pas de réordonnancement |
| Période intra | **32**, l'image intra est une vraie image de rafraîchissement (IDR) |
| Métrique | PSNR YUV (6:1:1) par image, moyenné ; débit en bpp, tout le flux compté |

Les quatre codecs :
- **x265** (HEVC, `preset medium`) : images P, 4 références, QP constant ;
- **VVenC** (VVC, Fraunhofer HHI) : configuration *low-delay* (B) officielle, profondeur interne 10 bits ;
- **ScaleSpaceFlow** (2020) : codage de **résidu** (Partie B) ;
- **DCVC-RT** (2025) : codage **conditionnel**. Le réseau ne calcule ni flot explicite ni résidu : l'image courante est codée **conditionnellement à un contexte temporel** appris extrait des images passées. Un seul modèle couvre 64 niveaux de qualité.

⏱️ Mode FAST sur T4 : environ 15 à 25 minutes au total. Chaque point est sauvegardé : en cas de coupure, relancez simplement les cellules.

In [ ]:
RES_DIR = os.path.join(WORK, "results", MODE)
os.makedirs(RES_DIR, exist_ok=True)
PARAMS = {"x265": [22, 27, 32, 37], "VVenC": [22, 27, 32, 37], "SSF": [2, 4, 6, 8], "DCVC-RT": [0, 21, 42, 63]}

def res_path(codec, seq, p):
    return os.path.join(RES_DIR, f"{codec}_{seq}_{p}.json")

def save_result(codec, seq, p, frames_bits, rec_frames, frame_types, t_enc=None, t_dec=None, extra=None):
    src = SEQ[seq]
    ps = np.array([psnr_yuv420(s, r) for s, r in zip(src, rec_frames)])
    res = dict(codec=codec, seq=seq, param=p, n=len(rec_frames),
               bits=[int(b) for b in frames_bits], type=list(frame_types),
               psnr_y=ps[:, 0].tolist(), psnr_u=ps[:, 1].tolist(), psnr_v=ps[:, 2].tolist(), psnr_yuv=ps[:, 3].tolist(),
               bpp=float(sum(frames_bits)) / (len(rec_frames) * W * H), psnr=float(ps[:, 3].mean()),
               t_enc=t_enc, t_dec=t_dec, **(extra or {}))
    assert len(frames_bits) == len(rec_frames) == N_FRAMES, "nombre d'images incohérent"
    with open(res_path(codec, seq, p), "w") as f: json.dump(res, f)
    return res

def done(codec, seq, p):
    return os.path.exists(res_path(codec, seq, p))

### C.1 — x265 (HEVC), low-delay P

In [ ]:
def run_x265(seq, qp, preset="medium"):
    src = SEQ_PATHS[seq]
    with tempfile.TemporaryDirectory() as d:
        bs, rec, csv = [os.path.join(d, f) for f in ("out.hevc", "rec.yuv", "frames.csv")]
        params = (f"qp={qp}:keyint={INTRA_PERIOD}:min-keyint={INTRA_PERIOD}:scenecut=0:bframes=0:ref=4:"
                  f"open-gop=0:csv={csv}:csv-log-level=1:log-level=error")
        t0 = time.time()
        sh(["ffmpeg", "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "yuv420p", "-s", f"{W}x{H}",
            "-r", str(FPS), "-i", src, "-frames:v", str(N_FRAMES), "-c:v", "libx265", "-preset", preset,
            "-x265-params", params, "-f", "hevc", bs])
        t_enc = time.time() - t0
        t0 = time.time()
        sh(["ffmpeg", "-y", "-loglevel", "error", "-i", bs, "-f", "rawvideo", "-pix_fmt", "yuv420p", rec])
        t_dec = time.time() - t0
        log = pd.read_csv(csv, skipinitialspace=True)
        log = log[pd.to_numeric(log["Encode Order"], errors="coerce").notna()].sort_values("Encode Order")
        bits = log["Bits"].astype(int).tolist()
        # les en-têtes (VPS/SPS/PPS/SEI) ne sont pas dans le CSV : on les ajoute à la première image
        bits[0] += 8 * os.path.getsize(bs) - sum(bits)
        return save_result("x265", seq, qp, bits, read_yuv420(rec, W, H, N_FRAMES), [t[0] for t in log["Type"]],
                           t_enc / N_FRAMES, t_dec / N_FRAMES)

if RUN["x265"]:
    for s in SEQUENCES:
        for qp in PARAMS["x265"]:
            if not done("x265", s, qp):
                r = run_x265(s, qp); print(f"x265 {s} QP{qp}: {r['bpp']:.4f} bpp, {r['psnr']:.2f} dB")

### C.2 — VVenC (VVC), low-delay B

VVenC est l'encodeur VVC open source du Fraunhofer HHI. On utilise sa configuration expérimentale `lowdelay_faster`, avec une période intra de 32 et des images IDR, comme dans les conditions de test du JVET.

In [ ]:
VVENC_TAG = "v1.14.0"
VVENC_BIN = os.path.join(WORK, "bin", "vvencFFapp")
VVENC_CFG = os.path.join(WORK, "bin", "lowdelay_faster.cfg")

def setup_vvenc(build_if_missing=True):
    if os.path.exists(VVENC_BIN) and os.path.exists(VVENC_CFG):
        return True
    b, c = hf_get("bin/vvencFFapp"), hf_get("bin/lowdelay_faster.cfg")
    if b and c:
        shutil.copy(b, VVENC_BIN); shutil.copy(c, VVENC_CFG); os.chmod(VVENC_BIN, 0o755)
        try:
            sh([VVENC_BIN, "--version"], check=False); return True
        except Exception as e:
            print("binaire VVenC inutilisable :", e)
    if not build_if_missing:
        return False
    print("Compilation de VVenC (~10 min, une seule fois : le binaire est gardé sur le Drive)…")
    src = os.path.join(LOCAL, "vvenc")
    if not os.path.exists(src):
        sh(["git", "clone", "-q", "--depth", "1", "--branch", VVENC_TAG, "https://github.com/fraunhoferhhi/vvenc.git", src])
    sh(f"cmake -S {src} -B {src}/build -DCMAKE_BUILD_TYPE=Release > /dev/null && cmake --build {src}/build -j{os.cpu_count()} --target vvencFFapp > /dev/null")
    found = glob.glob(f"{src}/bin/**/vvencFFapp", recursive=True)
    shutil.copy(found[0], VVENC_BIN); os.chmod(VVENC_BIN, 0o755)
    shutil.copy(f"{src}/cfg/experimental/lowdelay_faster.cfg", VVENC_CFG)
    return True

def run_vvenc(seq, qp):
    with tempfile.TemporaryDirectory() as d:
        bs, rec = os.path.join(d, "out.266"), os.path.join(d, "rec.yuv")
        t0 = time.time()
        out = sh([VVENC_BIN, "-c", VVENC_CFG, f"--InputFile={SEQ_PATHS[seq]}", f"--SourceWidth={W}", f"--SourceHeight={H}",
                  "--InputBitDepth=8", "--OutputBitDepth=8", "--InternalBitDepth=10", f"--FrameRate={FPS}",
                  f"--FramesToBeEncoded={N_FRAMES}", f"--QP={qp}", f"--IntraPeriod={INTRA_PERIOD}",
                  "--DecodingRefreshType=idr", f"--Threads={os.cpu_count()}", "--Verbosity=4",
                  f"--BitstreamFile={bs}", f"--ReconFile={rec}"])
        t_enc = time.time() - t0
        poc = {int(m.group(1)): (m.group(2), int(m.group(3)))
               for m in re.finditer(r"POC\s+(\d+)\s+TId:.*?([IPB])-SLICE.*?(\d+)\s+bits", out)}
        if len(poc) == N_FRAMES:
            types = [poc[i][0] for i in range(N_FRAMES)]; bits = [poc[i][1] for i in range(N_FRAMES)]
            bits[0] += 8 * os.path.getsize(bs) - sum(bits)        # en-têtes de paramètres
        else:   # format de journal inattendu : on répartit le débit total (la dérive par image sera perdue)
            print("  (bits par image non trouvés dans le journal VVenC)")
            types = ["I" if i % INTRA_PERIOD == 0 else "B" for i in range(N_FRAMES)]
            bits = [8 * os.path.getsize(bs) / N_FRAMES] * N_FRAMES
        return save_result("VVenC", seq, qp, bits, read_yuv420(rec, W, H, N_FRAMES), types, t_enc / N_FRAMES, None)

if RUN["VVenC"]:
    try:
        if setup_vvenc():
            for s in SEQUENCES:
                for qp in PARAMS["VVenC"]:
                    if not done("VVenC", s, qp):
                        r = run_vvenc(s, qp); print(f"VVenC {s} QP{qp}: {r['bpp']:.4f} bpp, {r['psnr']:.2f} dB")
    except Exception as e:
        print("VVenC indisponible :", str(e)[:500])

### C.3 — ScaleSpaceFlow

Même protocole : une image clé toutes les 32 images, et tout le flux réellement codé est compté. Les reconstructions RGB sont reconverties en YUV 4:2:0 pour la métrique.

In [ ]:
def run_ssf(seq, q):
    net = ssf2020(quality=q, pretrained=PRETRAINED).eval().to(device); net.update(force=True)
    bits, recs, types, t_enc, t_dec = [], [], [], 0.0, 0.0
    sync = (lambda: torch.cuda.synchronize()) if device == "cuda" else (lambda: None)
    with torch.no_grad():
        for g0 in range(0, N_FRAMES, INTRA_PERIOD):
            gop = [pad_to(yuv420_to_rgb(*f), PAD_SSF).to(device) for f in SEQ[seq][g0:g0 + INTRA_PERIOD]]
            sync(); t0 = time.time(); strings, shapes = net.compress(gop); sync(); t_enc += time.time() - t0
            t0 = time.time(); dec = net.decompress(strings, shapes); sync(); t_dec += time.time() - t0
            bits += [count_bits(s) for s in strings]
            recs += [rgb_to_yuv420(x[..., :H, :W]) for x in dec]
            types += ["I"] + ["P"] * (len(gop) - 1)
            del gop, dec
    del net
    if device == "cuda": torch.cuda.empty_cache()
    return save_result("SSF", seq, q, bits, recs, types, t_enc / N_FRAMES, t_dec / N_FRAMES)

if RUN["SSF"]:
    for s in SEQUENCES:
        for q in PARAMS["SSF"]:
            if not done("SSF", s, q):
                r = run_ssf(s, q); print(f"SSF {s} q{q}: {r['bpp']:.4f} bpp, {r['psnr']:.2f} dB")

### C.4 — DCVC-RT (Microsoft, CVPR 2025)

On utilise le **dépôt officiel** et son script de test, comme le ferait un ingénieur qui évalue un codec de recherche. Trois étapes :
1. cloner le dépôt à une version figée et compiler le **codeur entropique rANS** (C++/pybind11) ;
2. récupérer les **poids publics** (hébergés par Microsoft sur OneDrive, recopiés sur le dépôt de l'enseignant) ;
3. lancer `test_video.py` avec notre protocole, puis recalculer **nous-mêmes** les métriques sur les vidéos décodées.

Note de protocole : le script officiel écrit la chrominance décodée par **troncature** (et non par arrondi) ; nous corrigeons ce détail pour que tous les codecs soient évalués de la même façon.

In [ ]:
DCVC_COMMIT = "cbdae87a5445114cdc7f48816da63ea80bdeac40"
DCVC_ROOT = os.path.join(LOCAL, "DCVC")
RT_DIR = os.path.join(DCVC_ROOT, "DCVC-family", "DCVC-RT")
CKPT = {"i": os.path.join(WORK, "bin", "cvpr2025_image.pth.tar"), "p": os.path.join(WORK, "bin", "cvpr2025_video.pth.tar")}

def setup_dcvc():
    if device != "cuda":
        print("DCVC-RT exige un GPU CUDA."); return False
    if not os.path.exists(RT_DIR):
        sh(["git", "clone", "-q", "https://github.com/microsoft/DCVC.git", DCVC_ROOT])
        sh(["git", "-C", DCVC_ROOT, "checkout", "-q", DCVC_COMMIT])
    try:
        import MLCodec_extensions_cpp  # noqa: F401
    except ImportError:
        print("Compilation du codeur rANS…")
        sh([sys.executable, "-m", "pip", "install", "-q", "pybind11"])
        sh([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", os.path.join(RT_DIR, "src", "cpp")])
    for k, name in [("i", "cvpr2025_image.pth.tar"), ("p", "cvpr2025_video.pth.tar")]:
        if not os.path.exists(CKPT[k]):
            f = hf_get(f"checkpoints/dcvc_rt/{name}")
            if f is None:
                print(f"Poids {name} introuvables. Téléchargez-les depuis le README de DCVC-RT (OneDrive) et copiez-les dans {os.path.dirname(CKPT[k])}")
                return False
            shutil.copy(f, CKPT[k])
    # Correctif de protocole : arrondi (et non troncature) de la chrominance écrite
    tv = os.path.join(RT_DIR, "test_video.py")
    txt = open(tv).read()
    bad = "uv_rec = torch.clamp(uv_rec * 255, 0, 255).to(dtype=torch.uint8)"
    if bad in txt:
        open(tv, "w").write(txt.replace(bad, "uv_rec = torch.clamp(uv_rec * 255, 0, 255).round().to(dtype=torch.uint8)"))
    return True

def run_dcvc_all(seqs, qps, tag=None):
    todo = [(s, q) for s in seqs for q in qps if not done("DCVC-RT", s, q)]
    if not todo: return
    src_dir = os.path.join(LOCAL, "dcvc_src"); os.makedirs(src_dir, exist_ok=True)
    out_dir = os.path.join(LOCAL, "dcvc_out", tag or MODE)
    cfg = {"root_path": src_dir, "test_classes": {"UVG": {"test": 1, "base_path": "", "src_type": "yuv420", "sequences": {}}}}
    for s in seqs:
        link = os.path.join(src_dir, f"{s}_{W}x{H}")           # nom sans « .yuv » : chemins de sortie plus propres
        if not os.path.exists(link): os.symlink(SEQ_PATHS[s], link)
        cfg["test_classes"]["UVG"]["sequences"][f"{s}_{W}x{H}"] = {"width": W, "height": H, "frames": N_FRAMES, "intra_period": INTRA_PERIOD}
    cfg_path = os.path.join(LOCAL, "dcvc_cfg.json"); json.dump(cfg, open(cfg_path, "w"))
    cmd = [sys.executable, "test_video.py", "--model_path_i", CKPT["i"], "--model_path_p", CKPT["p"],
           "--rate_num", str(len(qps)), "--qp_i", *map(str, qps), "--qp_p", *map(str, qps),
           "--test_config", cfg_path, "--cuda", "1", "-w", "1", "--write_stream", "1", "--save_decoded_frame", "1",
           "--force_zero_thres", "0.12", "--force_intra_period", str(INTRA_PERIOD), "--reset_interval", "64",
           "--force_frame_num", str(N_FRAMES), "--check_existing", "1", "--verbose", "1", "--verbose_json", "1",
           "--stream_path", out_dir, "--output_path", os.path.join(out_dir, "summary.json")]
    print("Lancement de DCVC-RT…"); t0 = time.time()
    log = sh(cmd, cwd=RT_DIR)
    print(log[-600:]); print(f"(durée {time.time()-t0:.0f} s)")
    for s in seqs:
        for q in qps:
            base = os.path.join(out_dir, "UVG", f"{s}_{W}x{H}_q{q}")
            j = json.load(open(base + ".json"))
            rec = glob.glob(base + "_*kbps.yuv")[0]
            bits = [round(b * W * H) for b in j["frame_bpp"]]
            types = ["I" if t == 0 else "P" for t in j["frame_type"]]
            r = save_result("DCVC-RT", s, q, bits, read_yuv420(rec, W, H, N_FRAMES), types,
                            j.get("avg_frame_encoding_time"), j.get("avg_frame_decoding_time"),
                            extra=dict(psnr_official=j["ave_all_frame_psnr"], bin_bytes=os.path.getsize(base + ".bin")))
            print(f"DCVC-RT {s} qp{q}: {r['bpp']:.4f} bpp, {r['psnr']:.2f} dB (script officiel : {j['ave_all_frame_psnr']:.2f} dB)")

if RUN["DCVC-RT"]:
    try:
        if setup_dcvc():
            run_dcvc_all(SEQUENCES, PARAMS["DCVC-RT"])
    except Exception as e:
        print("DCVC-RT indisponible :", str(e)[:1500])

### C.5 — Rassemblement des résultats (et repli sur les résultats pré-calculés)

In [ ]:
def load_results():
    rows = [json.load(open(p)) for p in glob.glob(os.path.join(RES_DIR, "*.json"))]
    have = {(r["codec"], r["seq"]) for r in rows}
    if USE_PRECOMPUTED_IF_MISSING:
        f = hf_get(f"precomputed/tp2_{MODE}.jsonl")
        if f:
            for line in open(f):
                r = json.loads(line)
                if r["seq"] in SEQUENCES and (r["codec"], r["seq"]) not in have:
                    r["codec"] += " (pré-calculé)"; rows.append(r)
    return rows

ALL = load_results()
df = pd.DataFrame([{k: r[k] for k in ["codec", "seq", "param", "bpp", "psnr", "t_enc", "t_dec"]} for r in ALL])
print("Codecs disponibles :", sorted(df.codec.unique()))
df.sort_values(["seq", "codec", "bpp"]).round(4)

In [ ]:
STY = {"x265": "s--", "VVenC": "D--", "SSF": "^-", "DCVC-RT": "o-"}
style = lambda c: STY.get(c.replace(" (pré-calculé)", ""), "x-")
fig, axes = plt.subplots(1, len(SEQUENCES), figsize=(6.5 * len(SEQUENCES), 4.6), squeeze=False)
for ax, s in zip(axes[0], SEQUENCES):
    for c, g in df[df.seq == s].groupby("codec"):
        g = g.sort_values("bpp"); ax.plot(g.bpp, g.psnr, style(c), label=c, ms=5)
    ax.set_title(f"{s} — {W}x{H}, {N_FRAMES} images, IP={INTRA_PERIOD}, low-delay")
    ax.set_xlabel("bpp"); ax.set_ylabel("PSNR YUV 6:1:1 (dB)"); ax.grid(alpha=.3); ax.legend()
plt.tight_layout(); plt.show()

### C.6 — BD-rate

Même fonction qu'au TP1 : économie moyenne de débit à qualité égale (négatif = meilleur que l'ancre), calculée **par séquence** puis moyennée.

In [ ]:
def bd_rate(rate_anchor, q_anchor, rate_test, q_test):
    def interp(rate, q):
        q, lr = np.asarray(q, float), np.log(np.asarray(rate, float)); i = np.argsort(q)
        return PchipInterpolator(q[i], lr[i])
    lo, hi = max(min(q_anchor), min(q_test)), min(max(q_anchor), max(q_test))
    if hi <= lo: return np.nan
    fa, ft = interp(rate_anchor, q_anchor), interp(rate_test, q_test)
    return (np.exp((ft.integrate(lo, hi) - fa.integrate(lo, hi)) / (hi - lo)) - 1) * 100

def bd_table(anchor):
    out = {}
    for c in df.codec.unique():
        vals = {}
        for s in SEQUENCES:
            a, t = df[(df.codec == anchor) & (df.seq == s)], df[(df.codec == c) & (df.seq == s)]
            vals[s] = bd_rate(a.bpp, a.psnr, t.bpp, t.psnr) if len(a) >= 2 and len(t) >= 2 else np.nan
        vals["moyenne"] = np.nanmean(list(vals.values())) if any(np.isfinite(list(vals.values()))) else np.nan
        out[c] = vals
    return pd.DataFrame(out).T.round(1)

for anchor in ["x265", "VVenC"]:
    if anchor in set(df.codec):
        print(f"BD-rate PSNR YUV (%) — ancre : {anchor}"); display(bd_table(anchor))

### C.7 — Dérive de qualité au fil du GOP

Pour chaque codec, on prend le point de fonctionnement **le plus proche d'un débit cible** et on trace le PSNR et le débit **image par image**.

In [ ]:
TARGET_BPP = float(df[df.codec == "x265"].bpp.median()) if "x265" in set(df.codec) else float(df.bpp.median())
s = SEQUENCES[0]
fig, ax = plt.subplots(2, 1, figsize=(13, 7), sharex=True)
for c in sorted(df.codec.unique()):
    cand = [r for r in ALL if r["codec"] == c and r["seq"] == s]
    if not cand: continue
    r = min(cand, key=lambda r: abs(np.log(r["bpp"] / TARGET_BPP)))
    lab = f"{c} ({r['param']}, {r['bpp']:.3f} bpp)"
    ax[0].plot(r["psnr_yuv"], style(c), ms=3, label=lab)
    ax[1].plot(np.array(r["bits"]) / (W * H), style(c), ms=3, label=lab)
for g in range(0, N_FRAMES, INTRA_PERIOD):
    for a in ax: a.axvline(g, c="gray", ls=":", lw=1)
ax[0].set_ylabel("PSNR YUV (dB)"); ax[1].set_ylabel("bpp de l'image"); ax[1].set_yscale("log"); ax[1].set_xlabel("image")
ax[0].legend(fontsize=8, ncol=2); ax[0].grid(alpha=.3); ax[1].grid(alpha=.3)
ax[0].set_title(f"{s} : qualité et débit image par image (cible ≈ {TARGET_BPP:.3f} bpp)")
plt.tight_layout(); plt.show()

def drift(r):
    p = np.array(r["psnr_yuv"]); t = np.array(r["type"])
    first_p = p[1:min(5, INTRA_PERIOD)].mean(); last_p = p[max(1, min(INTRA_PERIOD, len(p)) - 4):min(INTRA_PERIOD, len(p))].mean()
    return first_p - last_p
print("Perte de PSNR entre le début et la fin de la première période intra (dB) :")
display(pd.DataFrame([dict(codec=r["codec"], seq=r["seq"], param=r["param"], bpp=r["bpp"], derive_dB=drift(r)) for r in ALL])
        .pivot_table(index="codec", columns="seq", values="derive_dB", aggfunc="mean").round(2))

### C.8 — Complexité

Attention : x265 et VVenC tournent sur **2 cœurs CPU** de Colab, les codecs neuronaux sur un **GPU**. Les temps ne sont comparables qu'avec cette précaution ; c'est justement une question centrale du déploiement.

In [ ]:
cx = df.groupby("codec")[["t_enc", "t_dec"]].mean().mul(1000).round(1).rename(columns={"t_enc": "encodage (ms/image)", "t_dec": "décodage (ms/image)"})
gpu_lbl = "GPU" if device == "cuda" else "CPU (pas de GPU !)"
cx["matériel"] = [gpu_lbl if any(k in c for k in ["SSF", "DCVC"]) else "CPU" for c in cx.index]
cx

**❓ Questions — Partie C**
9. Classez les quatre codecs par BD-rate. ScaleSpaceFlow fait-il mieux que x265 ? Et DCVC-RT par rapport à VVenC ? Qu'est-ce qui a changé entre 2020 et 2025 (architecture, **codage conditionnel** vs résidu, entraînement sur de longues séquences, modèle de contexte) ?
10. Pourquoi est-il **indispensable** de fixer la même période intra et la même structure (low-delay) pour tous les codecs ? Que deviendrait la comparaison avec une période intra de 10, comme dans certains articles ?
11. Décrivez la **dérive** de chaque codec le long du GOP. Expliquez pourquoi un codec neuronal entraîné sur des séquences courtes dérive davantage, et comment DCVC-RT et les codecs classiques la limitent (QP hiérarchique, adaptation des caractéristiques, rafraîchissement).
12. Le motif périodique du débit de VVenC (et de DCVC-RT) est-il visible ? À quoi correspond-il ?
13. Les métriques de DCVC-RT recalculées ici diffèrent légèrement de celles du script officiel. Identifiez les deux raisons (arrondi des reconstructions, chrominance). Quelle leçon en tirer pour un rapport de comparaison ?
14. À partir du tableau de complexité, discutez la phrase « DCVC-RT est plus rapide que VVenC ». Dans quelles conditions est-elle vraie ? Que faudrait-il pour décoder ce flux sur un téléphone (déterminisme, entiers, consommation) ?

---
## Pour aller plus loin
- **DCVC-FM** (CVPR 2024) et **DCVC-UF** (CVPR 2026) sont dans le même dépôt (`DCVC-family/`, racine du dépôt) : le premier ajoute la modulation des caractéristiques pour de très longues périodes intra, le second code des **paquets d'images** d'un coup.
- Remplacez `SEQUENCES` par vos propres vidéos (YUV 4:2:0) : toute la chaîne d'évaluation reste valable.
- Testez une période intra de **-1** (une seule image intra) en mode FULL : qui dérive le plus ?

## Rendu
Rapport d'équipe : courbes R-D, tableaux BD-rate et complexité, figure de dérive, réponses argumentées aux questions 1 à 9